# KPI 4 — Gross Margin Analysis

**requirements.md (Sales & Revenue, KPI 4):** "Gross Margin Analysis: Margin % per product/supplier; compare catalog price vs actual selling price."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_4_gross_margin` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cell below once (or after any change to the fact/dim/bridge definitions).

**Definitions (state formulas):**

* **Grain:** one row per **supplier × product pair** (`supplier_id`, `product_id`) — all 313 pairs in `bridge_supplier_products` sold on valid dates, so the view covers the full bridge; margin per product or per supplier = `GROUP BY` over this view (supplier-product ranking itself is KPI 14)
* **Revenue bases (KPI 1 convention):** gross = `SUM(line_total)` over ALL statuses with `order_date_valid = true`; net = same excluding `status = 'cancelled'`; totals reconcile exactly with KPI 1/2/3 (net 28,333,498.48 / gross 30,723,269.53)
* **COGS (net basis):** per line `COALESCE(line_total × supplier_price / unit_price, quantity × supplier_price)` — 445 net lines have null `quantity` (known silver issue; `quantity` and `unit_price` are never null together), so the `line_total`-based branch covers them; `supplier_price` resolves AS-OF the order header `created_at` against the SCD2 bridge, falling back to the pair's **first** version when the order predates the bridge (true for every order today: the bridge opened 2026-11-19, after the last order)
* **Gross profit** = net revenue − COGS; **margin %** = `ROUND(100 × gross_profit / net_revenue, 2)` — both net basis (cancelled orders earn no margin)
* **Catalog vs actual price:** `avg_list_price` = AVG of `dim_products.list_price` (the version as-of the sale, via the fact's as-of `product_sk`) over net lines; `avg_actual_unit_price` = AVG(`unit_price`); `avg_discount_pct` = `ROUND(100 × (list − actual) / list, 2)`
* `units_sold` = `SUM(quantity)` over net lines (null-quantity lines excluded); **below-cost data issue:** 3 pairs have 7 net lines sold under supplier cost (known silver issue, retained — losses are absorbed inside otherwise-profitable pairs; no pair has negative total margin)

Verification cells: shape & totals → independent per-pair recompute → supplier rollup → top pairs → below-cost pairs → category margin.

In [0]:
%sql
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_4_gross_margin
TBLPROPERTIES (
  'comment' = 'KPI 4 - Gross Margin Analysis (requirements.md: margin % per product/supplier; compare catalog price vs actual selling price). Grain: one row per supplier x product pair (all 313 bridge pairs sold on valid dates; margin per product/supplier = GROUP BY over this view). Gross = SUM(line_total) ALL statuses, order_date_valid = true; net = same excluding status = ''cancelled'' (KPI 1 bases). COGS (net basis) per line = COALESCE(line_total * supplier_price / unit_price, quantity * supplier_price) - 445 net lines with null quantity covered by the line_total branch; supplier_price AS-OF order created_at from the SCD2 bridge, first-version fallback when the order predates the bridge. Gross profit = net - COGS; margin % = ROUND(100 * gross_profit / net, 2). avg_list_price = AVG(dim_products.list_price as-of the sale) and avg_actual_unit_price = AVG(unit_price) over net lines; avg_discount_pct = ROUND(100 * (list - actual) / list, 2). units_sold = SUM(quantity) net lines. below_cost_line_count flags the known below-cost data issue.'
)
AS
WITH lines AS (
  SELECT
    f.order_id,
    f.order_created_at,
    f.status,
    f.quantity,
    f.unit_price,
    f.line_total,
    d.product_id,
    d.list_price,
    s.supplier_id
  FROM b2b_ecommerce.gold.fact_order_items f
  JOIN b2b_ecommerce.gold.dim_products  d ON f.product_sk  = d.product_sk
  JOIN b2b_ecommerce.gold.dim_suppliers  s ON f.supplier_sk = s.supplier_sk
  WHERE f.order_date_valid
),
priced AS (
  SELECT
    l.*,
    COALESCE(a.supplier_price, fv.supplier_price) AS supplier_price
  FROM lines l
  LEFT JOIN b2b_ecommerce.gold.bridge_supplier_products a
    ON  a.supplier_id = l.supplier_id
    AND a.product_id = l.product_id
    AND a.__START_AT <= l.order_created_at
    AND (a.__END_AT IS NULL OR l.order_created_at < a.__END_AT)
  LEFT JOIN (
    SELECT b.supplier_id, b.product_id, b.supplier_price
    FROM b2b_ecommerce.gold.bridge_supplier_products b
    JOIN (
      SELECT supplier_id, product_id, MIN(__START_AT) AS ms
      FROM b2b_ecommerce.gold.bridge_supplier_products
      GROUP BY supplier_id, product_id
    ) m
      ON  b.supplier_id = m.supplier_id
      AND b.product_id  = m.product_id
      AND b.__START_AT  = m.ms
  ) fv
    ON  fv.supplier_id = l.supplier_id
    AND fv.product_id  = l.product_id
),
per_pair AS (
  SELECT
    p.supplier_id,
    p.product_id,
    SUM(p.line_total)                                                    AS gross_rev,
    SUM(CASE WHEN p.status <> 'cancelled' THEN p.line_total ELSE 0 END)   AS net_rev,
    SUM(CASE WHEN p.status <> 'cancelled'
             THEN COALESCE(p.line_total * p.supplier_price / p.unit_price,
                           p.quantity * p.supplier_price)
             ELSE 0 END)                                                  AS cogs_val,
    SUM(CASE WHEN p.status <> 'cancelled' THEN p.quantity END)            AS units,
    AVG(CASE WHEN p.status <> 'cancelled' THEN p.list_price END)          AS avg_list,
    AVG(CASE WHEN p.status <> 'cancelled' THEN p.unit_price END)          AS avg_actual,
    SUM(CASE WHEN p.status <> 'cancelled' AND p.unit_price < p.supplier_price
             THEN 1 ELSE 0 END)                                           AS below_cost_lines,
    COUNT(DISTINCT p.order_id)                                            AS order_cnt,
    COUNT(*)                                                             AS line_cnt
  FROM priced p
  GROUP BY p.supplier_id, p.product_id
),
attributed AS (
  SELECT
    pp.supplier_id,
    cc.name           AS supplier_name,
    pp.product_id,
    dp.sku            AS sku,
    dp.name           AS product_name,
    dp.category       AS category,
    pp.gross_rev,
    pp.net_rev,
    pp.cogs_val,
    pp.units,
    pp.avg_list,
    pp.avg_actual,
    pp.below_cost_lines,
    pp.order_cnt,
    pp.line_cnt
  FROM per_pair pp
  LEFT JOIN b2b_ecommerce.gold.dim_suppliers  s  ON s.supplier_id = pp.supplier_id
  LEFT JOIN b2b_ecommerce.gold.dim_companies   cc ON cc.company_id = s.company_id AND cc.__END_AT IS NULL
  LEFT JOIN b2b_ecommerce.gold.dim_products    dp ON dp.product_id = pp.product_id AND dp.__END_AT IS NULL
)
SELECT
  supplier_id,
  supplier_name,
  product_id,
  sku,
  product_name,
  category,
  gross_rev                                              AS gross_revenue,
  net_rev                                                AS net_revenue,
  ROUND(cogs_val, 2)                                     AS cogs,
  ROUND(net_rev - cogs_val, 2)                           AS gross_profit,
  ROUND(100 * ROUND(net_rev - cogs_val, 2) / net_rev, 2) AS gross_margin_pct,
  units                                                  AS units_sold,
  order_cnt                                              AS order_count,
  line_cnt                                               AS line_count,
  ROUND(avg_list, 2)                                     AS avg_list_price,
  ROUND(avg_actual, 2)                                   AS avg_actual_unit_price,
  ROUND(100 * (avg_list - avg_actual) / avg_list, 2)      AS avg_discount_pct,
  below_cost_lines                                       AS below_cost_line_count
FROM attributed;

In [0]:
%sql
-- Verification: shape, totals, reconciliation. Expected: 313 rows = 313 distinct
-- supplier-product pairs (= full bridge, all sold on valid dates), 25 distinct suppliers
-- x 160 distinct products, SUM(net) = 28,333,498.48 and SUM(gross) = 30,723,269.53
-- (exact KPI 1/2/3 reconciliation), SUM(cogs) = 20,725,008.21, SUM(gross_profit) =
-- 7,608,490.27, overall margin 26.85%, 0 null names/sku/category/cogs/margin,
-- 0 negative gross_profit, 3 pairs with below-cost lines (7 lines total).
SELECT
  COUNT(*)                                      AS rows_pairs,
  COUNT(DISTINCT CONCAT(supplier_id, '|', product_id)) AS distinct_pairs,
  COUNT(DISTINCT supplier_id)                    AS distinct_suppliers,
  COUNT(DISTINCT product_id)                     AS distinct_products,
  SUM(gross_revenue)                             AS sum_gross,
  SUM(net_revenue)                               AS sum_net,
  SUM(cogs)                                      AS sum_cogs,
  SUM(gross_profit)                              AS sum_gross_profit,
  ROUND(100 * SUM(gross_profit) / SUM(net_revenue), 2) AS overall_margin_pct,
  SUM(CASE WHEN supplier_name IS NULL OR sku IS NULL OR product_name IS NULL
            OR category IS NULL THEN 1 ELSE 0 END)  AS null_supplier_or_product_attrs,
  SUM(CASE WHEN cogs IS NULL OR gross_margin_pct IS NULL THEN 1 ELSE 0 END) AS null_cogs_or_margin,
  SUM(CASE WHEN gross_profit < 0 THEN 1 ELSE 0 END) AS negative_margin_pairs,
  SUM(CASE WHEN below_cost_line_count > 0 THEN 1 ELSE 0 END) AS below_cost_pairs,
  SUM(below_cost_line_count)                     AS below_cost_lines
FROM b2b_ecommerce.gold.kpi_4_gross_margin;

In [0]:
%sql
-- Verification: independently recompute every pair from the raw fact via the OPEN-version
-- bridge join (a different path than the view's AS-OF + first-version join; identical today
-- because each pair has exactly one version), and re-derive margin %, gross profit and
-- discount % from the view's own columns. Also reconcile units with the fact net lines.
-- Expected: pairs_checked = 313, every match count = 313, units_view = units_fact.
WITH recompute AS (
  SELECT
    s.supplier_id,
    d.product_id,
    SUM(f.line_total)                                                     AS gross_rev,
    SUM(CASE WHEN f.status <> 'cancelled' THEN f.line_total ELSE 0 END)    AS net_rev,
    SUM(CASE WHEN f.status <> 'cancelled'
             THEN COALESCE(f.line_total * b.supplier_price / f.unit_price,
                           f.quantity * b.supplier_price)
             ELSE 0 END)                                                    AS cogs,
    SUM(CASE WHEN f.status <> 'cancelled' THEN f.quantity END)            AS units,
    AVG(CASE WHEN f.status <> 'cancelled' THEN d.list_price END)           AS avg_list,
    AVG(CASE WHEN f.status <> 'cancelled' THEN f.unit_price END)           AS avg_actual
  FROM b2b_ecommerce.gold.fact_order_items f
  JOIN b2b_ecommerce.gold.dim_products  d ON f.product_sk  = d.product_sk
  JOIN b2b_ecommerce.gold.dim_suppliers  s ON f.supplier_sk = s.supplier_sk
  LEFT JOIN b2b_ecommerce.gold.bridge_supplier_products b
    ON  b.supplier_id = s.supplier_id
    AND b.product_id  = d.product_id
    AND b.__END_AT IS NULL
  WHERE f.order_date_valid
  GROUP BY s.supplier_id, d.product_id
),
checks AS (
  SELECT
    v.supplier_id, v.product_id,
    v.gross_revenue = r.gross_rev                                      AS gross_match,
    v.net_revenue   = r.net_rev                                        AS net_match,
    v.cogs          = ROUND(r.cogs, 2)                                AS cogs_match,
    v.units_sold    = r.units                                         AS units_match,
    v.avg_list_price = ROUND(r.avg_list, 2)                           AS list_match,
    v.avg_actual_unit_price = ROUND(r.avg_actual, 2)                  AS actual_match,
    v.gross_profit  = ROUND(r.net_rev - r.cogs, 2)                    AS profit_match,
    v.gross_margin_pct = ROUND(100 * ROUND(r.net_rev - r.cogs, 2) / r.net_rev, 2) AS margin_match,
    v.avg_discount_pct = ROUND(100 * (r.avg_list - r.avg_actual) / r.avg_list, 2) AS discount_match
  FROM b2b_ecommerce.gold.kpi_4_gross_margin v
  JOIN recompute r
    ON  r.supplier_id = v.supplier_id
    AND r.product_id  = v.product_id
)
SELECT
  COUNT(*)                                        AS pairs_checked,
  SUM(CASE WHEN gross_match    THEN 1 ELSE 0 END) AS gross_revenue_match,
  SUM(CASE WHEN net_match      THEN 1 ELSE 0 END) AS net_revenue_match,
  SUM(CASE WHEN cogs_match     THEN 1 ELSE 0 END) AS cogs_match,
  SUM(CASE WHEN units_match    THEN 1 ELSE 0 END) AS units_sold_match,
  SUM(CASE WHEN list_match     THEN 1 ELSE 0 END) AS avg_list_price_match,
  SUM(CASE WHEN actual_match   THEN 1 ELSE 0 END) AS avg_actual_price_match,
  SUM(CASE WHEN profit_match   THEN 1 ELSE 0 END) AS gross_profit_match,
  SUM(CASE WHEN margin_match   THEN 1 ELSE 0 END) AS margin_pct_match,
  SUM(CASE WHEN discount_match THEN 1 ELSE 0 END) AS discount_pct_match,
  (SELECT SUM(units_sold) FROM b2b_ecommerce.gold.kpi_4_gross_margin) AS units_view,
  (SELECT SUM(quantity) FROM b2b_ecommerce.gold.fact_order_items
    WHERE order_date_valid AND status <> 'cancelled')                 AS units_fact
FROM checks;

In [0]:
%sql
-- Margin per supplier (rollup over the view): 25 suppliers ordered by net revenue,
-- with each supplier's share of total net revenue, margin % and below-cost lines.
SELECT
  supplier_id,
  supplier_name,
  COUNT(*)                                              AS pairs,
  SUM(net_revenue)                                      AS net_revenue,
  SUM(gross_profit)                                     AS gross_profit,
  ROUND(100 * SUM(gross_profit) / SUM(net_revenue), 2)  AS gross_margin_pct,
  ROUND(100 * SUM(net_revenue) / SUM(SUM(net_revenue)) OVER (), 2) AS net_revenue_share_pct,
  SUM(below_cost_line_count)                            AS below_cost_lines
FROM b2b_ecommerce.gold.kpi_4_gross_margin
GROUP BY supplier_id, supplier_name
ORDER BY net_revenue DESC;

In [0]:
%sql
-- Top 10 supplier-product pairs by gross profit.
SELECT supplier_id, supplier_name, sku, product_name, category,
       net_revenue, cogs, gross_profit, gross_margin_pct,
       units_sold, avg_list_price, avg_actual_unit_price, avg_discount_pct
FROM b2b_ecommerce.gold.kpi_4_gross_margin
ORDER BY gross_profit DESC
LIMIT 10;

In [0]:
%sql
-- The 3 pairs containing the known below-cost lines (sold under supplier cost;
-- losses absorbed - no pair has negative total margin).
SELECT supplier_id, supplier_name, sku, product_name, category,
       net_revenue, cogs, gross_profit, gross_margin_pct,
       below_cost_line_count, avg_actual_unit_price
FROM b2b_ecommerce.gold.kpi_4_gross_margin
WHERE below_cost_line_count > 0
ORDER BY gross_profit DESC;

In [0]:
%sql
-- Category margin rollup (margin per product = GROUP BY product over the view).
SELECT
  category,
  COUNT(*)                                              AS pairs,
  SUM(net_revenue)                                      AS net_revenue,
  SUM(gross_profit)                                     AS gross_profit,
  ROUND(100 * SUM(gross_profit) / SUM(net_revenue), 2)  AS gross_margin_pct,
  ROUND(AVG(avg_discount_pct), 2)                       AS avg_discount_pct
FROM b2b_ecommerce.gold.kpi_4_gross_margin
GROUP BY category
ORDER BY gross_profit DESC;